# Análise de dados de crédito


#### Contexto
Analisando dados de uma empresa do setor financeiro 
que opera cartões Private Label.

Nos últimos meses, a empresa observou: 
1. Aumento relevante na inadimplência da carteira 
2. Dúvidas sobre a qualidade das informações de renda dos clientes 
3. Pressão da diretoria para expandir a base de clientes e aumentar receita, 
sem elevar o risco de forma descontrolada

A base de dados contém essas informações para apoiar uma 
análise de revisão da política de crédito e, em decorrência das dúvidas sobre 
qualidade na informação de renda, um enriquecimento obtido através de um 
backtest do produto “Renda Presumida” de um Bureau de Crédito. 

**Objetivo**

O objetivo é analisar a base disponibilizada e desenvolver um diagnóstico 
aprofundado da carteira de clientes, com foco na identificação dos principais 
fatores que influenciam o risco de crédito e na avaliação da qualidade das 
informações disponíveis.

### Configurações do ambiente

#### Instalação das bibliotecas

In [ ]:
!python.exe -m pip install --upgrade pip
%pip install pandas numpy pyarrow matplotlib seaborn openpyxl duckdb


#### Importando bibliotecas e definindo aliases

In [ ]:
import pandas as pd
import numpy as np
import pyarrow
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb

### Base de clientes

#### Carregamento e leitura da planilha da base de clientes em carteira

In [ ]:
clientes = pd.read_excel("infos_clientes.xlsx")

Visualização prévia da tabela

In [ ]:
clientes.head()

In [ ]:
clientes.describe()

In [ ]:
clientes.dtypes

In [ ]:
clientes.isnull().sum()

#### Tratamentos da tabela de clientes

Limpando o cabeçalho da tabela

In [ ]:
clientes.columns = clientes.columns.str.strip()

Preenchendo valores nulos com 0

In [ ]:
clientes = clientes.fillna(0)

Convertendo a coluna 'data_de_nascimento' (str) em Date

In [ ]:
clientes['data_de_nascimento'] = pd.to_datetime(
    clientes['data_de_nascimento'],
    format='%d/%m/%Y',
    errors='coerce'
)

clientes['data_de_nascimento']

Removendo microssegundos das colunas de datetime

In [ ]:
datas = [
    'Data_Cadastro_Cliente',
    'Data_Cadastro_Cartao',
    'Data_1_Compra',
    'Venc_1_Compra'
]

clientes[datas] = clientes[datas].apply(
    lambda col: col.dt.floor('s')
)

clientes[datas]

#### Testando hipóteses e fazendo validações

##### Perfil demográfico

Vamos analisar a distribuição de idades dos clientes no geral

In [ ]:
hoje = pd.Timestamp.today()

clientes['idade_atual'] = (
    (
        hoje - clientes['data_de_nascimento']
    ).dt.days // 365
)


plt.figure(figsize=(12,6))

bins = np.arange(
    clientes['idade_atual'].min(),
    clientes['idade_atual'].max() + 5,
    5
)

clientes['idade_atual'].plot(
    kind='hist',
    bins=bins,
    alpha=0.7,
    edgecolor='black'
)

plt.title('Distribuição de Idade dos Clientes')
plt.xlabel('Idade')
plt.ylabel('Quantidade')

plt.xticks(bins)

plt.show()

Percebe-se que existem clientes bem distribuidos entre diferentes idade, sendo que a maioria possui aproximadamente 30 - 45 anos de idade. Não estão concentrados nas extremidades.

Mas eles figuram entre os maiores inadimplentes? Vejamos:

Filtrando clientes com Status PROTESTO e INADIMPLENTE

In [ ]:
filtro = clientes[
    clientes['Status']
    .str.upper()
    .isin(['PROTESTO', 'INADIMPLENTE'])
]

Validando os indices de inadimplência por faixa etária.

In [ ]:
clientes['faixa_etaria'] = pd.cut(
    clientes['idade_atual'],
    bins=[0,18,25,35,45,60,100],
    labels=[
        '0-18',
        '19-25',
        '26-35',
        '36-45',
        '46-60',
        '60+'
    ]
)

filtro['faixa_etaria'] = pd.cut(
    filtro['idade_atual'],
    bins=[0,18,25,35,45,60,100],
    labels=[
        '0-18',
        '19-25',
        '26-35',
        '36-45',
        '46-60',
        '60+'
    ]
)

# total por faixa etária
total_faixa = (
    clientes['faixa_etaria']
    .value_counts()
    .sort_index()
)

# inadimplentes por faixa etária
inadimplentes_faixa = (
    filtro['faixa_etaria']
    .value_counts()
    .reindex(total_faixa.index)
    .fillna(0)
)

# percentual
percentual = (
    inadimplentes_faixa / total_faixa * 100
).round(2)

# dataframe comparativo
comparativo = pd.DataFrame({
    'Total Clientes': total_faixa,
    'Inadimplentes': inadimplentes_faixa
})

# gráfico
fig, ax1 = plt.subplots(figsize=(14,7))

# barras lado a lado
comparativo.plot(
    kind='bar',
    ax=ax1
)

ax1.set_title('Clientes vs Inadimplência por Faixa Etária')
ax1.set_xlabel('Faixa Etária')
ax1.set_ylabel('Quantidade')

# rótulos nas barras
for container in ax1.containers:
    ax1.bar_label(
        container,
        fmt='%.0f',
        padding=3
    )

# eixo secundário
ax2 = ax1.twinx()

# linha percentual
percentual.plot(
    kind='line',
    marker='o',
    color='red',
    linewidth=2,
    ax=ax2
)

ax2.set_ylabel('Percentual (%)')

# rótulos da linha
for i, valor in enumerate(percentual):
    ax2.text(
        i,
        valor,
        f'{valor:.2f}%',
        color='red',
        ha='center',
        va='bottom'
    )

# legendas fora do gráfico
ax1.legend(
    loc='upper left',
    bbox_to_anchor=(1.02, 1)
)

ax2.legend(
    ['Percentual'],
    loc='upper left',
    bbox_to_anchor=(1.02, 0.9)
)

plt.tight_layout()
plt.show()

Confirmado. Além de serem maioria na carteira de clientes, os maiores percentuais de inadimplência estão nessa parcela de clientes.

Podemos verificar também quanto a situação dos clientes. Temos suspeitas de que os clientes desempregados ou sem emprego fixo podem encontrar dificuldades para manter as contas em dia.
Vejamos se confere:

In [ ]:
# total por situação
total_situacao = (
    clientes['situacao']
    .value_counts()
    .sort_values(ascending=False)
)

# inadimplentes por situação
inadimplentes_situacao = (
    filtro['situacao']
    .value_counts()
    .reindex(total_situacao.index)
    .fillna(0)
)

# percentual
percentual = (
    inadimplentes_situacao / total_situacao * 100
).round(2)

# dataframe comparativo
comparativo = pd.DataFrame({
    'Total Clientes': total_situacao,
    'Inadimplentes': inadimplentes_situacao
})

# gráfico
fig, ax1 = plt.subplots(figsize=(16,7))

# barras lado a lado
comparativo.plot(
    kind='bar',
    ax=ax1
)

ax1.set_title('Clientes vs Inadimplência por Situação')
ax1.set_xlabel('Situação')
ax1.set_ylabel('Quantidade')

# rótulos barras
for container in ax1.containers:
    ax1.bar_label(
        container,
        fmt='%.0f',
        padding=3
    )

# eixo secundário
ax2 = ax1.twinx()

# linha percentual
percentual.plot(
    kind='line',
    marker='o',
    color='red',
    linewidth=2,
    ax=ax2
)

ax2.set_ylabel('Percentual (%)')

# rótulos linha
for i, valor in enumerate(percentual):
    ax2.text(
        i,
        valor,
        f'{valor:.2f}%',
        color='red',
        ha='center',
        va='bottom'
    )

# legendas externas
ax1.legend(
    loc='upper left',
    bbox_to_anchor=(1.02, 1)
)

ax2.legend(
    ['Percentual'],
    loc='upper left',
    bbox_to_anchor=(1.02, 0.9)
)

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

Interessante. Na verdade os maiores inadimplentes empatados são os clientes que declararam serem registrados e pensionistas, seguidos pelos autônomos.
Podemos supor que os demais preferem não fazer muitas dívidas justamente pela incerteza de suas situações instáveis, em contraste com a maioria apresentada.

Os desempregados representam uma parcela muito pequena da carteira, possivelmente por não conseguirem crédito, visto que, não geram renda a ser comprovada.

Outro dado disponível é o UF (estado dos clientes), vejamos em qual região se encontra os maiores índices.

Segmentando as regiões

In [ ]:
mapa_regiao = {
    # Norte
    'AC': 'Norte',
    'AP': 'Norte',
    'AM': 'Norte',
    'PA': 'Norte',
    'RO': 'Norte',
    'RR': 'Norte',
    'TO': 'Norte',

    # Nordeste
    'AL': 'Nordeste',
    'BA': 'Nordeste',
    'CE': 'Nordeste',
    'MA': 'Nordeste',
    'PB': 'Nordeste',
    'PE': 'Nordeste',
    'PI': 'Nordeste',
    'RN': 'Nordeste',
    'SE': 'Nordeste',

    # Centro-Oeste
    'DF': 'Centro-Oeste',
    'GO': 'Centro-Oeste',
    'MT': 'Centro-Oeste',
    'MS': 'Centro-Oeste',

    # Sudeste
    'ES': 'Sudeste',
    'MG': 'Sudeste',
    'RJ': 'Sudeste',
    'SP': 'Sudeste',

    # Sul
    'PR': 'Sul',
    'RS': 'Sul',
    'SC': 'Sul'
}

clientes['regiao'] = clientes['uf'].map(mapa_regiao)
filtro['regiao'] = clientes['uf'].map(mapa_regiao)

In [ ]:
total_regiao = (
    clientes['regiao']
    .value_counts()
    .sort_values(ascending=False)
)

inadimplentes_regiao = (
    filtro['regiao']
    .value_counts()
    .reindex(total_regiao.index)
    .fillna(0)
)

percentual = (
    inadimplentes_regiao / total_regiao * 100
).round(2)

comparativo = pd.DataFrame({
    'Total Clientes': total_regiao,
    'Inadimplentes': inadimplentes_regiao
})

# gráfico
fig, ax1 = plt.subplots(figsize=(14,7))

comparativo.plot(
    kind='bar',
    ax=ax1
)

ax1.set_title('Clientes vs Inadimplência por Região')
ax1.set_xlabel('Região')
ax1.set_ylabel('Quantidade')

# rótulos nas barras
for container in ax1.containers:
    ax1.bar_label(
        container,
        fmt='%.0f',
        padding=3
    )

# eixo secundário
ax2 = ax1.twinx()

percentual.plot(
    kind='line',
    marker='o',
    color='red',
    linewidth=2,
    ax=ax2
)

ax2.set_ylabel('Percentual (%)')

# rótulos na linha
for i, valor in enumerate(percentual):
    ax2.text(
        i,
        valor,
        f'{valor:.2f}%',
        color='red',
        ha='center',
        va='bottom'
    )

# legenda
ax1.legend(
    loc='upper left',
    bbox_to_anchor=(1.02, 1)
)

ax2.legend(
    ['Percentual'],
    loc='upper left',
    bbox_to_anchor=(1.02, 0.9)
)

plt.subplots_adjust(right=0.9)
plt.show()

Os clientes do Nordeste e Sudeste são maioria. Porém, em termos de inadimplência apesar ser menos representativa em volume, proporcionalmente a região Sul lidera em localização dos mais inadimplentes.

##### Perfil financeiro

A variavel de pontuação de score é uma das mais analisadas. Pois mede exatamente o comportamento financeiro do indivíduo. Podemos ver se há alguma ligação com indices altos de inadimplência.

In [ ]:
plt.figure(figsize=(12,6))

bins = np.arange(0, 1001, 100)

clientes['score'].plot(
    kind='hist',
    bins=bins,
    edgecolor='black'
)

plt.title('Distribuição de Clientes por Score')
plt.xlabel('Score')
plt.ylabel('Quantidade')

plt.xticks(bins)

plt.show()

In [ ]:
clientes['faixa_score'] = pd.cut(
    filtro['score'],
    bins=[0,300,500,700,1000],
    labels=[
        'Muito Baixo',
        'Baixo',
        'Médio',
        'Alto'
    ]
)

filtro['faixa_score'] = pd.cut(
    filtro['score'],
    bins=[0,300,500,700,1000],
    labels=[
        'Muito Baixo',
        'Baixo',
        'Médio',
        'Alto'
    ]
)

# totais por score
total_score = (
    clientes['faixa_score']
    .value_counts()
    .sort_index()
)

# inadimplentes por score
inadimplentes_score = (
    filtro['faixa_score']
    .value_counts()
    .reindex(total_score.index)
    .fillna(0)
)

# percentual
percentual = (
    inadimplentes_score / total_score * 100
).round(2)

# dataframe comparativo
comparativo = pd.DataFrame({
    'Total Clientes': total_score,
    'Inadimplentes': inadimplentes_score
})

# gráfico
fig, ax1 = plt.subplots(figsize=(14,7))

comparativo.plot(
    kind='bar',
    ax=ax1
)

ax1.set_title('Clientes vs Inadimplência por Score')
ax1.set_xlabel('Faixa de Score')
ax1.set_ylabel('Quantidade')

# rótulos nas barras
for container in ax1.containers:
    ax1.bar_label(
        container,
        fmt='%.0f',
        padding=3
    )

# eixo secundário
ax2 = ax1.twinx()

# linha percentual
percentual.plot(
    kind='line',
    marker='o',
    color='red',
    linewidth=2,
    ax=ax2
)

ax2.set_ylabel('Percentual (%)')

# rótulos da linha
for i, valor in enumerate(percentual):
    ax2.text(
        i,
        valor,
        f'{valor:.2f}%',
        color='red',
        ha='center',
        va='bottom'
    )

# legendas fora
ax1.legend(
    loc='upper left',
    bbox_to_anchor=(1.02, 1)
)

ax2.legend(
    ['Percentual'],
    loc='upper left',
    bbox_to_anchor=(1.02, 0.9)
)

plt.subplots_adjust(right=0.85)

plt.show()

De fato quanto menor a faixa do score em que o cliente está enquadrado maior a taxa de inadimplência.

### Base de movimentações de cartões

#### Carregamento e Leitura do parquet de movimentações de cartões (compras e pagamentos/ recebimento)

In [ ]:
movimentacoes = pd.read_parquet("base_compras_pagamentos.parquet", engine="pyarrow")

In [ ]:
movimentacoes

#### Tratamento da base de movimentações

Limpando o cabeçalho da tabela

In [ ]:
movimentacoes.columns = movimentacoes.columns.str.strip()

Preenchendo os valores nulos com 0

In [ ]:
movimentacoes = movimentacoes.fillna(0)
movimentacoes

#### Testando hipóteses e fazendo validações

In [ ]:
### Clientes com valores pagos mas sem valor de compra informado
pagto_sem_compra_informada = movimentacoes[movimentacoes['Valor_Compra']<=0]['Ctrl_ClienteCartao'].count()

### Clientes com valores pagos maiores que o valor de compra informado
pagto_maiores_que_compras = movimentacoes[movimentacoes['Valor_Compra']<movimentacoes['Valor_Recebimento']]['Ctrl_ClienteCartao'].count()

print("Pagto. sem compra infomada: ",pagto_sem_compra_informada)
print("Pagto. mais altos que o valor da compra: ",pagto_maiores_que_compras)



Pode-se observar que há movimentações de pagamentos que não apresentam o valor da compra. 
Possivelmente, não estavam previstos e/ou a informação do valor de compra se perdeu. 

Também, é possível identificar dentre as movimentações, valores pagos maiores que o valor da compra.
Podendo se tratar de juros e/ou multas sobre atraso mas não está explícito, visto que os valores estão agregados e não possuímos a informação de data do pagamento para validar se foi realizado após o vencimento e se sim, quanto tempo depois, se foi integral ou parcialmente quitada.

### Cruzamento de dados - Merge das Tabelas

Verificação de tipos das chaves para join das tabelas

In [ ]:
clientes['ID_Cartao'].dtype
movimentacoes['Ctrl_ClienteCartao'].dtype

Criação de métricas de movimentações

In [ ]:
metricas = (
    movimentacoes.groupby('Ctrl_ClienteCartao')
       .agg(
           qtde_compras=('Ctrl_ClienteCartao', 'count'),
           valor_em_compras=('Valor_Compra', 'sum'),
           ticket_medio=('Valor_Compra', 'mean'),
           valor_recebimento=('Valor_Recebimento', 'sum'),
       )
       .reset_index()
)

clientes_mov = clientes.merge(
    metricas,
    left_on='ID_Cartao',
    right_on='Ctrl_ClienteCartao',
    how='left'
)

clientes_mov.head()

In [ ]:
qtde_clientes = clientes_mov['ID_Cliente'].count()
clientes_sem_mov = clientes_mov[clientes_mov['qtde_compras'] < 0]['ID_Cliente'].count()

print("Quantidade total de clientes: ", qtde_clientes)
print("Quantidade de clientes sem movimentação: ", clientes_sem_mov)


In [ ]:
df_corr = clientes_mov.drop(
    ['ID_Cliente', 'PDV_Vendedor', 'ID_Cartao', 'Ctrl_ClienteCartao'],
    axis=1,
    errors='ignore'
)

df_corr = df_corr.select_dtypes(include='number').dropna()

corr = df_corr.corr()

plt.figure(figsize=(10,6))

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Mapa de Correlação entre Variáveis')

plt.show()